In [5]:
import os
import mesa
import time
import json
from google import genai
from google.genai import types

api_key = os.environ.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)

class ProfilingAgent(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)

    def step(self):
        prompt = f"You are Agent {self.unique_id}. Respond STRICTLY with JSON: {{'status': 'active'}}"
        
        start_time = time.time()
        try:
            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt,
                config=types.GenerateContentConfig(response_mime_type="application/json")
            )
            _ = json.loads(response.text)
            status = "Success"
        except Exception as e:
            status = "Failed"
            
        end_time = time.time()
        
        latency = end_time - start_time
        self.model.record_latency(self.unique_id, self.model.steps, latency, status)
        time.sleep(4)

class ProfilingModel(mesa.Model):
    def __init__(self, n_agents):
        super().__init__()
        self.latency_log = []
        
        for i in range(n_agents):
            agent = ProfilingAgent(self)
            
    def record_latency(self, agent_id, step, latency, status):
        self.latency_log.append({
            "Agent": agent_id,
            "Step": step,
            "Latency (s)": round(latency, 3),
            "Status": status
        })
        
    def step(self):
        self.agents.shuffle_do("step")
        
    def generate_report(self):
        # Generate a production-grade terminal report
        print("\n" + "="*55)
        print("🚀 MESA-LLM NEURO-SYMBOLIC LATENCY BENCHMARK")
        print("="*55)
        print(f"{'Agent ID':<10} | {'Step':<8} | {'Latency (sec)':<15} | {'Status'}")
        print("-" * 55)
        
        total_time = 0
        success_count = 0
        
        for entry in self.latency_log:
            print(f"{entry['Agent']:<10} | {entry['Step']:<8} | {entry['Latency (s)']:<15} | {entry['Status']}")
            if entry['Status'] == "Success":
                total_time += entry['Latency (s)']
                success_count += 1
                
        print("-" * 55)
        if success_count > 0:
            avg = round(total_time / success_count, 3)
            print(f"⏱️  AVERAGE API LATENCY: {avg} seconds / agent")
        print("="*55)

In [6]:
benchmark_model = ProfilingModel(3)

print("Starting benchmark. Please wait (throttling API calls to prevent 429 errors)...")
for _ in range(2):
    benchmark_model.step()

benchmark_model.generate_report()

Starting benchmark. Please wait (throttling API calls to prevent 429 errors)...

🚀 MESA-LLM NEURO-SYMBOLIC LATENCY BENCHMARK
Agent ID   | Step     | Latency (sec)   | Status
-------------------------------------------------------
2          | 1        | 2.087           | Success
1          | 1        | 1.607           | Success
3          | 1        | 1.909           | Success
2          | 2        | 1.447           | Success
3          | 2        | 2.952           | Success
1          | 2        | 3.241           | Success
-------------------------------------------------------
⏱️  AVERAGE API LATENCY: 2.207 seconds / agent
